In [0]:
# ============================================================
# GOLD 6.2 — Crude-Nifty Daily Correlation
# ============================================================
# Daily Brent vs Nifty returns with conflict-period segmentation.
# KPIs: Pearson correlation, sign-reversal rate.
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG iran_israel_capstone_project")

silver = spark.table("silver.daily_market_clean")
events = spark.table("silver.event_dim")

# ============================================================
# 1. CONFLICT WINDOW FLAG
# ============================================================
# Collect HIGH/CRITICAL event dates
hc_dates = (
    events
    .filter(F.col("severity").isin("HIGH", "CRITICAL"))
    .select(F.col("event_date"))
    .distinct()
)

# For each trading day, check if ANY HIGH/CRITICAL event is
# within 20 calendar days. Use a range join.
daily = (
    silver
    .select(
        "trade_date",
        "nifty_close", "nifty_daily_return_pct",
        "brent_close", "brent_daily_change_pct",
    )
    .filter(
        F.col("nifty_daily_return_pct").isNotNull()
        & F.col("brent_daily_change_pct").isNotNull()
    )
)

# Cross join then check date proximity
conflict_flags = (
    daily.alias("d")
    .crossJoin(hc_dates.alias("e"))
    .filter(
        (F.datediff(F.col("d.trade_date"), F.col("e.event_date")) >= 0)
        & (F.datediff(F.col("d.trade_date"), F.col("e.event_date")) <= 20)
    )
    .select(F.col("d.trade_date"))
    .distinct()
)

# Join flag back onto daily data
gold_corr = (
    daily
    .join(conflict_flags, on="trade_date", how="left_semi")
    .withColumn("in_conflict_window", F.lit(True))
).unionByName(
    daily
    .join(conflict_flags, on="trade_date", how="left_anti")
    .withColumn("in_conflict_window", F.lit(False))
).orderBy("trade_date")

# Add sign-reversal flag: Brent UP and Nifty DOWN
gold_corr = gold_corr.withColumn(
    "sign_reversal",
    (F.col("brent_daily_change_pct") > 0)
    & (F.col("nifty_daily_return_pct") < 0)
)

# ============================================================
# 2. PEARSON CORRELATION by conflict window
# ============================================================
corr_df = (
    gold_corr
    .groupBy("in_conflict_window")
    .agg(
        F.count("*").alias("trading_days"),
        F.round(
            F.corr("brent_daily_change_pct", "nifty_daily_return_pct"), 4
        ).alias("pearson_r"),
        F.round(F.avg("brent_daily_change_pct"), 4).alias("avg_brent_chg"),
        F.round(F.avg("nifty_daily_return_pct"), 4).alias("avg_nifty_ret"),
    )
    .orderBy("in_conflict_window")
)

print("\n" + "="*60)
print("  PEARSON CORRELATION: Brent vs Nifty Daily Returns")
print("="*60)
corr_df.show(truncate=False)

corr_rows = {r["in_conflict_window"]: r for r in corr_df.collect()}
conflict_r  = corr_rows[True]["pearson_r"]
quiet_r     = corr_rows[False]["pearson_r"]
stronger    = "CONFLICT" if abs(conflict_r) > abs(quiet_r) else "NON-CONFLICT"

print(f"  Conflict  Pearson r : {conflict_r:+.4f}  ({corr_rows[True]['trading_days']} days)")
print(f"  Quiet     Pearson r : {quiet_r:+.4f}  ({corr_rows[False]['trading_days']} days)")
print(f"  Stronger |r| in     : {stronger} period")

# ============================================================
# 3. SIGN REVERSAL RATE (Brent UP & Nifty DOWN)
# ============================================================
reversal_df = (
    gold_corr
    .groupBy("in_conflict_window")
    .agg(
        F.count("*").alias("total_days"),
        F.sum(F.col("sign_reversal").cast("int")).alias("reversal_days"),
    )
    .withColumn(
        "reversal_pct",
        F.round(F.col("reversal_days") / F.col("total_days") * 100, 2)
    )
    .orderBy("in_conflict_window")
)

print("\n" + "="*60)
print("  SIGN REVERSAL RATE (Brent ↑ & Nifty ↓)")
print("="*60)
reversal_df.show(truncate=False)

rev_rows = {r["in_conflict_window"]: r for r in reversal_df.collect()}
conflict_rev = rev_rows[True]["reversal_pct"]
quiet_rev    = rev_rows[False]["reversal_pct"]
rev_pass     = conflict_rev > quiet_rev
rev_status   = "✅" if rev_pass else "❌"

print(f"  Conflict  reversal rate : {conflict_rev:.2f}%")
print(f"  Quiet     reversal rate : {quiet_rev:.2f}%")
print(f"  Rule (conflict > quiet) : {rev_status}")
print("="*60)

# ============================================================
# 4. PERSIST TO GOLD
# ============================================================
spark.sql("CREATE SCHEMA IF NOT EXISTS iran_israel_capstone_project.gold")

(
    gold_corr.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("iran_israel_capstone_project.gold.gold_crude_nifty_daily_correlation")
)

row_count = gold_corr.count()
print(f"\n✅ gold.gold_crude_nifty_daily_correlation written ({row_count} rows)")
display(gold_corr)